# One-dimensional convolution with shared memory

This notebook compares a direct convolution kernel with a tiled version that loads the signal and its halos into shared memory. It first prints a small boundary case. It then runs two controlled timing experiments: a block-size sweep at fixed signal length, and a larger working set at fixed block size. Every GPU result is checked against a CPU reference.


## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the available GPU and CUDA compiler.

In [ ]:
!nvidia-smi
!nvcc --version
!mkdir -p code


## CUDA error and timing support

`CUDA_CHECK` handles CUDA errors independently of `TimingGPU`. The timer uses CUDA events; measured intervals include the kernel launches on the default stream, but exclude allocation, host-device copies, and CPU validation.

In [ ]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


In [ ]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


## Kernels and host program

The filter width is odd, and samples outside the signal are treated as zero. The direct kernel loads each window from global memory. The shared kernel cooperatively loads a tile of `blockDim.x + 2*radius` values, including both halos, and synchronizes before computing valid outputs. All threads reach the barrier. The filter coefficients stay in global memory in both versions.


In [ ]:
%%writefile code/chapter02_convolution_shared.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>

const int DEFAULT_THREADS = 256;
const int REPEATS = 50;
const int BATCHES = 5;

__global__ void convolution_direct(const float *signal, const float *filter,
                                  float *result, int n, int k) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        const int radius = k / 2;
        float sum = 0.0f;
        for (int j = 0; j < k; ++j) {
            const int input_index = i + j - radius;
            if (input_index >= 0 && input_index < n)
                sum += signal[input_index] * filter[j];
        }
        result[i] = sum;
    }
}

__global__ void convolution_shared(const float *signal, const float *filter,
                                  float *result, int n, int k) {
    extern __shared__ float tile[];
    const int radius = k / 2;
    const int tile_size = blockDim.x + 2 * radius;
    const int block_start = blockIdx.x * blockDim.x;

    // Threads cooperatively load the block segment and both halos.
    for (int q = threadIdx.x; q < tile_size; q += blockDim.x) {
        const int input_index = block_start - radius + q;
        tile[q] = (input_index >= 0 && input_index < n)
                    ? signal[input_index] : 0.0f;
    }
    __syncthreads();

    const int i = block_start + threadIdx.x;
    if (i < n) {
        float sum = 0.0f;
        for (int j = 0; j < k; ++j)
            sum += tile[threadIdx.x + j] * filter[j];
        result[i] = sum;
    }
}

void launch_direct(const float *signal, const float *filter, float *result,
                   int n, int k, int threads) {
    const int blocks = (n + threads - 1) / threads;
    convolution_direct<<<blocks, threads>>>(signal, filter, result, n, k);
    CUDA_CHECK(cudaGetLastError());
}

void launch_shared(const float *signal, const float *filter, float *result,
                   int n, int k, int threads) {
    const int blocks = (n + threads - 1) / threads;
    const size_t shared_bytes = size_t(threads + k - 1) * sizeof(float);
    convolution_shared<<<blocks, threads, shared_bytes>>>(
        signal, filter, result, n, k);
    CUDA_CHECK(cudaGetLastError());
}

void make_filter(float *filter, int k) {
    float sum = 0.0f;
    for (int j = 0; j < k; ++j) {
        filter[j] = float(j + 1);
        sum += filter[j];
    }
    for (int j = 0; j < k; ++j) filter[j] /= sum;
}

void make_signal(float *signal, int n) {
    for (int i = 0; i < n; ++i)
        signal[i] = 0.6f * std::sin(0.013f * float(i))
                  + 0.3f * std::cos(0.007f * float(i));
}

void make_cpu_reference(const float *signal, const float *filter,
                        float *reference, int n, int k) {
    const int radius = k / 2;
    for (int i = 0; i < n; ++i) {
        float sum = 0.0f;
        for (int j = 0; j < k; ++j) {
            const int input_index = i + j - radius;
            if (input_index >= 0 && input_index < n)
                sum += signal[input_index] * filter[j];
        }
        reference[i] = sum;
    }
}

void check_result(const float *gpu, const float *reference,
                  int n, const char *name) {
    for (int i = 0; i < n; ++i) {
        const float tolerance = 2.0e-5f * (1.0f + std::fabs(reference[i]));
        if (std::fabs(gpu[i] - reference[i]) > tolerance) {
            std::fprintf(stderr,
                "%s mismatch at i=%d: CPU=%.8f GPU=%.8f\n",
                name, i, reference[i], gpu[i]);
            std::exit(EXIT_FAILURE);
        }
    }
}

void run_case(int n, int k, int threads, bool print_values, bool measure_time) {
    if (k <= 0 || (k % 2) == 0) {
        std::fprintf(stderr, "Filter width must be a positive odd number.\n");
        std::exit(EXIT_FAILURE);
    }
    const size_t signal_bytes = size_t(n) * sizeof(float);
    const size_t filter_bytes = size_t(k) * sizeof(float);
    float *h_signal = (float*)std::malloc(signal_bytes);
    float *h_filter = (float*)std::malloc(filter_bytes);
    float *h_reference = (float*)std::malloc(signal_bytes);
    float *h_direct = (float*)std::malloc(signal_bytes);
    float *h_shared = (float*)std::malloc(signal_bytes);
    float *d_signal = nullptr, *d_filter = nullptr;
    float *d_direct = nullptr, *d_shared = nullptr;
    if (!h_signal || !h_filter || !h_reference || !h_direct || !h_shared) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    make_signal(h_signal, n);
    make_filter(h_filter, k);
    make_cpu_reference(h_signal, h_filter, h_reference, n, k);

    CUDA_CHECK(cudaMalloc((void**)&d_signal, signal_bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_filter, filter_bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_direct, signal_bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_shared, signal_bytes));
    CUDA_CHECK(cudaMemcpy(d_signal, h_signal, signal_bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_filter, h_filter, filter_bytes, cudaMemcpyHostToDevice));

    launch_direct(d_signal, d_filter, d_direct, n, k, threads);
    launch_shared(d_signal, d_filter, d_shared, n, k, threads);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_direct, d_direct, signal_bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_shared, d_shared, signal_bytes, cudaMemcpyDeviceToHost));
    check_result(h_direct, h_reference, n, "Direct kernel");
    check_result(h_shared, h_reference, n, "Shared-memory kernel");
    std::printf("N=%d, K=%d, threads/block=%d: CPU/direct PASS; CPU/shared PASS\n",
                n, k, threads);

    if (print_values) {
        std::printf(" i    signal[i]       CPU      direct      shared\n");
        for (int i = 0; i < n; ++i)
            std::printf("%2d %11.6f %11.6f %11.6f %11.6f\n",
                        i, h_signal[i], h_reference[i], h_direct[i], h_shared[i]);
    }

    if (measure_time) {
        for (int warmup = 0; warmup < 10; ++warmup) {
            launch_direct(d_signal, d_filter, d_direct, n, k, threads);
            launch_shared(d_signal, d_filter, d_shared, n, k, threads);
        }
        CUDA_CHECK(cudaDeviceSynchronize());
        TimingGPU timer;
        float direct_ms[BATCHES] = {};
        float shared_ms[BATCHES] = {};
        for (int batch = 0; batch < BATCHES; ++batch) {
            const bool shared_first = (batch % 2) != 0;
            for (int position = 0; position < 2; ++position) {
                const bool use_shared = shared_first ? (position == 0) : (position == 1);
                timer.StartCounter();
                for (int repeat = 0; repeat < REPEATS; ++repeat) {
                    if (use_shared)
                        launch_shared(d_signal, d_filter, d_shared, n, k, threads);
                    else
                        launch_direct(d_signal, d_filter, d_direct, n, k, threads);
                }
                const float ms = timer.GetCounter() / REPEATS;
                if (use_shared) shared_ms[batch] = ms;
                else            direct_ms[batch] = ms;
            }
        }
        float direct_mean = 0.0f, shared_mean = 0.0f;
        for (int batch = 0; batch < BATCHES; ++batch) {
            direct_mean += direct_ms[batch];
            shared_mean += shared_ms[batch];
        }
        direct_mean /= BATCHES;
        shared_mean /= BATCHES;
        std::printf("Mean over %d batches of %d launches; kernel order alternated.\n",
                    BATCHES, REPEATS);
        std::printf("Direct: %.6f ms/launch; shared: %.6f ms/launch; direct/shared: %.3f\n",
                    direct_mean, shared_mean, direct_mean / shared_mean);
    }

    CUDA_CHECK(cudaFree(d_signal)); CUDA_CHECK(cudaFree(d_filter));
    CUDA_CHECK(cudaFree(d_direct)); CUDA_CHECK(cudaFree(d_shared));
    std::free(h_signal); std::free(h_filter); std::free(h_reference);
    std::free(h_direct); std::free(h_shared);
}

int main() {
    // Small case: prints all values and exposes both signal boundaries.
    run_case(13, 5, DEFAULT_THREADS, true, false);

    // Experiment 1: vary block size at fixed N and filter width.
    const int block_sizes[] = {128, 256, 512};
    const int n_fixed = 1 << 20;
    const int filter_widths[] = {15, 63};
    for (int k : filter_widths) {
        for (int threads : block_sizes)
            run_case(n_fixed, k, threads, false, true);
    }

    // Experiment 2: increase the working set while keeping block/filter fixed.
    run_case(1 << 24, 15, DEFAULT_THREADS, false, true);
    return EXIT_SUCCESS;
}


## Compile and run

The first case prints CPU, direct-GPU, and shared-GPU values for `N=13`, `K=5`, including the boundaries. The first timing experiment fixes `N=2^20` and compares block sizes 128, 256, and 512 for `K=15` and `K=63`. The second uses `N=2^24` with `K=15` and 256 threads/block to test a larger working set; the signal and one output array each occupy 64 MiB, although whether they exceed L2 depends on the GPU. Every case checks both GPU results against the CPU. Timings average five batches of 50 launches, with the direct/shared order alternated between batches. A ratio `direct/shared` above 1 means shared memory was faster in that run; no outcome is assumed.


In [ ]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_convolution_shared.cu code/TimingGPU.cu -o convolution_demo
./convolution_demo
